In [4]:
!pip install -U datasets transformers torch

In [14]:
from datasets import load_dataset
import torch
import torch.nn.functional as F


In [6]:
# Ma’lumotlar to‘plamini yuklash va tayyorlash

dataset = load_dataset('sst2')

small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(2000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(400))])

In [10]:
# Matnni tokenizatsiya qilish

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['sentence'], truncation=True, padding='max_length',max_length=128)

tokenzied_train = small_train.map(preprocess_function, batched=True)
tokenzied_test = small_test.map(preprocess_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [15]:
#  Modelni shug‘ullantirish

from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
    logging_dir="./logs",
    report_to = 'none')

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenzied_train,
    eval_dataset=tokenzied_test)

trainer.train()
trainer.evaluate()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but n

Step,Training Loss


KeyboardInterrupt: 

In [13]:
# Modelni baholash va sinovdan o‘tkazish

texts = [
    "this movie is a masterpiece",
    "a complete waste of time",

]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt",max_length=128).to('cuda')

with torch.no_grad():
  outputs = model(**inputs)
  probs = F.softmax(outputs.logits, dim=1)
  predictions = torch.argmax(probs, dim=1)

label_map = {0: 'negative', 1: 'positive'}

for text, pred,prob in zip(texts, predictions,probs):
  print(text,
        label_map[pred.item()],
        prob[pred.item()].item(),
        )


AssertionError: Torch not compiled with CUDA enabled